# SmolVLM base: causal perturbation and object localization

Enable Internet and a GPU (T4 x2 is supported). Attach the existing SmolVLM feature
cache. **Push the complete implementation and frozen data folder before using cell 1.**
Cell 1 clones/pulls Visualize, clears old package imports, and preserves outputs.

2,581 fixed test questions; greedy max 64 new tokens; FP32; initial batch 16/GPU,
CUDA OOM halves it. Top/Bottom/Random-32 (three Random repeats), ranking layers
1/7/13/19/25 (zero-based). Fixed generated answer is teacher-forced after zeroing
post-connector visual embeddings. Scores use geometric-mean token probabilities.

Localization uses both fixed partitions (749/652 targets), exact case-sensitive
literal phrases in actual generated output, global peak over 384 positions,
and inclusive bbox grid-center hits in six five-layer groups. Missing/repeated/
ambiguous/unalignable targets are audited. This is not segmentation accuracy.

Run new/partial measurements in order. **Redraw complete results: cell 1, then
cells 8–11 only**; these cells do not load a model or scan features. Only the few
raw camera images selected for overlays are downloaded, serially and cached.
Interrupting a wait cell leaves workers running; rerun that cell to reconnect.
Keep the whole output folder to resume. Changing measurement identity requires
a new OUTPUT; plotting-only changes do not require new measurement.


In [ ]:
# 1. Push changes first. Pull is blocked while any attention worker runs.
import os
from pathlib import Path
SOURCE_REPOSITORY = 'https://github.com/huyvanzzz/Log_Metrics.git'
SOURCE_BRANCH = 'smolvlm-attention'
SOURCE = Path('/kaggle/working/visualize-github-source')
OUTPUT = Path('/kaggle/working/smolvlm_probing_outputs')
RUN_MODE = 'auto'  # auto, measure, resume, plot
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HOME'] = '/kaggle/working/.cache/huggingface'
os.environ['PIP_CACHE_DIR'] = '/kaggle/working/.cache/pip'

"""Standard-library-only Git setup, embedded verbatim in both Kaggle notebooks."""
import hashlib
import importlib
import importlib.util
import io
import json
import os
from pathlib import Path
import subprocess
import sys
import zipfile


def ensure_no_workers(output_roots):
    for name in ('smolvlm_attention.runtime', 'smolvlm_attention.generated_runtime',
                 'dinov2_attention.runtime', 'smolvlm_probing.runtime'):
        module = sys.modules.get(name)
        if module is not None and any(process.poll() is None for process in getattr(module, '_PROCESSES', {}).values()):
            raise RuntimeError('A measurement worker is still running. Wait for it to finish before pulling source.')
    # The canonical probing output is shared by all notebooks. Also inspect it
    # after a kernel restart, when the new runtime is no longer in sys.modules.
    roots = set(map(Path, output_roots)) | {Path('/kaggle/working/smolvlm_probing_outputs')}
    for root in roots:
        for path in (Path(root) / 'logs').glob('*.pid.json'):
            pid = json.loads(path.read_text(encoding='utf-8'))['pid']
            try:
                os.kill(pid, 0)
            except (OSError, ProcessLookupError):
                continue
            raise RuntimeError(f'A measurement worker PID {pid} may still be running ({path}). Finish it before pulling.')


def sync_source(repository, branch, directory, packages=('smolvlm_inference', 'smolvlm_attention')):
    directory = Path(directory).resolve()
    def git(*args):
        return subprocess.check_output(['git', '-C', str(directory), *args], text=True).strip()
    if directory.exists():
        if not (directory / '.git').exists():
            raise RuntimeError(f'{directory} is not a Git checkout. Choose a new SOURCE directory.')
        if git('remote', 'get-url', 'origin').rstrip('/') != str(repository).rstrip('/'):
            raise RuntimeError('Existing source checkout has a different origin; choose a new SOURCE directory.')
        if git('status', '--porcelain', '--untracked-files=no'):
            raise RuntimeError('Source checkout has local changes. Preserve them before pulling; no files were reset.')
        if git('branch', '--show-current') != branch:
            raise RuntimeError('Checkout branch differs from SOURCE_BRANCH. Use the matching branch or a new SOURCE directory.')
        subprocess.run(['git', '-C', str(directory), 'pull', '--ff-only', 'origin', branch], check=True)
    else:
        directory.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', branch,
                        str(repository), str(directory)], check=True)
    # Keep the historical source_archive_sha256 field reproducible without embedding stale source.
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for package in packages:
            folder = directory / package
            if not folder.is_dir():
                raise RuntimeError(f'Missing source package after pull: {package}')
            for path in sorted(folder.iterdir()):
                if path.suffix in ('.py', '.txt'):
                    entry = zipfile.ZipInfo(path.relative_to(directory).as_posix(), (2026, 1, 1, 0, 0, 0))
                    entry.compress_type = zipfile.ZIP_DEFLATED
                    archive.writestr(entry, path.read_bytes())
    return dict(commit=git('rev-parse', 'HEAD'), sha256=hashlib.sha256(buffer.getvalue()).hexdigest())


def activate_source(directory, packages=('smolvlm_inference', 'smolvlm_attention')):
    directory = Path(directory).resolve()
    for package in packages:
        for path in (directory / package).glob('*.py'):
            Path(importlib.util.cache_from_source(str(path))).unlink(missing_ok=True)
    for name in list(sys.modules):
        if any(name == package or name.startswith(package + '.')
               for package in packages):
            del sys.modules[name]
    sys.path[:] = [str(directory)] + [entry for entry in sys.path if entry != str(directory)]
    importlib.invalidate_caches()

ensure_no_workers([OUTPUT, Path('/kaggle/working/attention_outputs'),
                   Path('/kaggle/working/generated_attention_outputs'),
                   Path('/kaggle/working/dinov2_attention_outputs')])
identity = sync_source(SOURCE_REPOSITORY, SOURCE_BRANCH, SOURCE,
                       packages=('smolvlm_probing', 'smolvlm_inference', 'smolvlm_attention'))
SOURCE_COMMIT, SOURCE_SHA256 = identity['commit'], identity['sha256']
activate_source(SOURCE, packages=('smolvlm_probing', 'smolvlm_inference', 'smolvlm_attention'))
from smolvlm_probing.runtime import resolve_run_mode
BOOT_MODE = resolve_run_mode(OUTPUT, RUN_MODE)
if BOOT_MODE != 'plot':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=tokenizers',
                    '-r', str(SOURCE/'smolvlm_probing/requirements.txt')], check=True)
    import torch, transformers, tokenizers
    if transformers.__version__ != '4.46.3' or tokenizers.__version__ != '0.20.4':
        raise RuntimeError('Restart the kernel after installing pinned dependencies, then rerun cell 1.')
RUN_CONFIG = OUTPUT/'probing_run.json'
print(dict(source=str(SOURCE), commit=SOURCE_COMMIT, output=str(OUTPUT), mode=BOOT_MODE,
           model='SmolVLM-256M base', initial_batch_per_gpu=16))


In [ ]:
# 2. Existing mounted cache/prepared paths are optional; automatic discovery is the default.
from smolvlm_probing.runtime import prepare_run, launch_stage, wait_stage, load_results
from smolvlm_probing.state import atomic_json
CACHE_INPUT = ''  # Set only if more than one matching cache is mounted.
PREPARED_INPUT = None  # Existing directory with manifest/train/val/test, otherwise pinned HF files.
print(dict(mode=BOOT_MODE, cache=CACHE_INPUT or 'auto', prepared=PREPARED_INPUT or 'pinned HF'))


In [ ]:
# 3. Validate fixed dataset/cache once, reuse matching receipt, and prepare all 2,581 prompts.
if BOOT_MODE != 'plot':
    RUN_CONFIG = prepare_run(OUTPUT, mode=BOOT_MODE, cache_input_dir=CACHE_INPUT,
                             prepared_dir=PREPARED_INPUT, source_commit=SOURCE_COMMIT,
                             source_sha256=SOURCE_SHA256)
print(RUN_CONFIG)


In [ ]:
# 4. Strict loading, native eager/scoring/zero-ablation and cached-decode preflight.
if BOOT_MODE != 'plot':
    launch_stage(RUN_CONFIG, 'preflight')
    wait_stage(RUN_CONFIG, 'preflight')


In [ ]:
# 5. Generate once, save exact original IDs, then collect baseline ranks and target maps.
if BOOT_MODE != 'plot':
    launch_stage(RUN_CONFIG, 'baseline')
    wait_stage(RUN_CONFIG, 'baseline')


In [ ]:
# 6. Causal: 26 perturbation conditions per nonempty valid answer; no new generation.
if BOOT_MODE != 'plot':
    launch_stage(RUN_CONFIG, 'causal')
    wait_stage(RUN_CONFIG, 'causal')


In [ ]:
# 7. CPU localization over saved target maps. Same eligible denominator for all six groups.
if BOOT_MODE != 'plot':
    launch_stage(RUN_CONFIG, 'localization')
    wait_stage(RUN_CONFIG, 'localization')


In [ ]:
# 8. Load complete checked results and draw five causal scatter figures.
from smolvlm_probing.runtime import load_results
from smolvlm_probing.plotting import plot_causal, plot_localization, select_examples
from smolvlm_probing.plotting import fetch_example_images, plot_examples, bundle_figures
from smolvlm_probing.state import atomic_json
RESULTS = load_results(RUN_CONFIG, plot_source_commit=SOURCE_COMMIT, plot_source_sha256=SOURCE_SHA256)
FIGURES = OUTPUT/'figures'
FIGURE_PATHS = plot_causal(RESULTS['causal'], FIGURES, metadata=RESULTS['metadata'])
print(RESULTS['metadata'])


In [ ]:
# 9. Accuracy by layer group, both coordinate partitions, with coverage and counts.
FIGURE_PATHS += plot_localization(RESULTS['summary'], FIGURES, metadata=RESULTS['metadata'])
print(RESULTS['summary'])


In [ ]:
# 10. Post-hoc examples from each partition's best layer group; fetch ONLY these image paths.
EXAMPLES = select_examples(RESULTS['localization'], RESULTS['summary'], seed=42)
atomic_json(OUTPUT/'selected_examples.json', EXAMPLES)
IMAGE_PATHS, MISSING_IMAGES = fetch_example_images(EXAMPLES, OUTPUT/'illustration_images')
atomic_json(OUTPUT/'missing_images.json', MISSING_IMAGES)
FIGURE_PATHS += plot_examples(EXAMPLES, IMAGE_PATHS, FIGURES, metadata=RESULTS['metadata'])
print(dict(missing_images=MISSING_IMAGES, figures=[str(p) for p in FIGURE_PATHS]))


In [ ]:
# 11. ZIP figures/captions/summary/audit/provenance; no checkpoint, features or source-image archive.
extras = [OUTPUT/name for name in ('summary.json', 'causal_summary.json', 'figure_metadata.json',
          'localization_audit.json', 'selected_examples.json', 'missing_images.json')]
extras += list(FIGURES.glob('*_caption.txt')) + list(FIGURES.glob('*_metadata.json'))
ZIP = bundle_figures(OUTPUT, FIGURE_PATHS, RESULTS['metadata'], extra_paths=extras)
from IPython.display import FileLink, display
display(FileLink(str(ZIP)))
